# AI Lab - Practical Question No. 10
## State-Space Visualizer for Reflex Agent

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Visualization & State-Space Formulation
- **Core AI Concept**: State-Space Formulation, Transition Model, Goal States, Cyclic Transitions
- **Python Concepts Tested**: Combinatorial state generation (`itertools`), NetworkX directed graphs (`DiGraph` & `MultiDiGraph`), Matplotlib publication-grade plotting
- **Complexity**: Time Complexity: $O(|S|^2)$ | Space Complexity: $O(|S|^2)$
- **Optional Challenge**: Highlight the goal states (both rooms clean) in green.

---

### Problem Statement
Implement a Python script that models a vacuum agent's state space.
For a 2-room environment, there are 8 possible states.
Generate all 8 states mathematically, link them via valid actions (`Left`, `Right`, `Suck`), and visualize this state-space graph using `matplotlib` and `networkx`.

### Task Requirements
1. **Generate the 8 states**: Represented as $(\text{AgentLoc}, \text{RoomA\_Status}, \text{RoomB\_Status})$.
2. **Define valid transitions**: Mathematical mapping $\text{Result}(s, a)$ for actions $\{\text{Left}, \text{Right}, \text{Suck}\}$.
3. **Create a directed graph**: Using `networkx` (`DiGraph` / `MultiDiGraph`).
4. **Plot the graph**: Using `matplotlib` with clear node/edge labels and legends.
5. **Constraints / Conditions**: Correctly identify cyclic transitions (e.g., `Left` when in Room A).
6. **Optional Challenge**: Visually highlight the goal states (both rooms clean) in emerald green.

### 1. Mathematical State-Space Formulation

#### State Space Size Formula
For an environment with $n$ distinct locations, where each location can independently be in one of $k$ cleanliness states:
$$\left|S\right| = n \times k^n$$

For the classic 2-room vacuum world ($n = 2, k = 2$):
$$\left|S\right| = 2 \times 2^2 = 2 \times 4 = 8\text{ states}$$

#### Canonical States (Russell & Norvig AIMA Convention):
| State ID | Agent Location | Room A Status | Room B Status | Goal State? |
| :--- | :--- | :--- | :--- | :--- |
| **S1** | A | Dirty | Dirty | No |
| **S2** | B | Dirty | Dirty | No |
| **S3** | A | Clean | Dirty | No |
| **S4** | B | Clean | Dirty | No |
| **S5** | A | Dirty | Clean | No |
| **S6** | B | Dirty | Clean | No |
| **S7** | A | Clean | Clean | **YES (Goal)** |
| **S8** | B | Clean | Clean | **YES (Goal)** |

#### Action Set and Transitions:
Actions $A = \{\text{Left}, \text{Right}, \text{Suck}\}$.
Total transitions across all states = $8 \times 3 = 24$ directed transitions.

- **Cyclic Transitions (Self-Loops)**: Occur when $\text{Result}(s, a) = s$.
  - `Left` at Room A produces 4 self-loops (States 1, 3, 5, 7).
  - `Right` at Room B produces 4 self-loops (States 2, 4, 6, 8).
  - `Suck` in a clean room produces 4 self-loops (States 3, 6, 7, 8).
  - Total: Exactly **12 self-loops** (50%) and **12 state-changing transitions** (50%).

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path when running from notebooks/
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import networkx as nx

from src.state_space_visualizer import (
    ACTION_LEFT,
    ACTION_RIGHT,
    ACTION_SUCK,
    COLOR_GOAL,
    COLOR_NON_GOAL,
    StateSpace,
    StateSpaceVisualizer,
    VacuumState,
    get_viva_reference,
)

print("Modules imported successfully.")

### 2. Algorithmic State Generation & Transition Mapping

In [ ]:
# Initialize mathematical state space model
state_space = StateSpace()

# Verify state count and list all 8 states
print(f"Total States Generated: {len(state_space.states)}")
for s in state_space.states:
    goal_str = " [GOAL STATE: Both Clean]" if s.is_goal else ""
    print(f"State {s.state_id}: Loc={s.agent_loc}, A={s.room_a_status:<5}, B={s.room_b_status:<5}{goal_str}")

### 3. Transition Table & Cyclic Transitions Audit

In [ ]:
visualizer = StateSpaceVisualizer(state_space)

# Display comprehensive ASCII transition matrix and self-loop statistics
print(visualizer.render_ascii_graph())

### 4. Cycle Detection & Self-Loop Analysis

In [ ]:
cycle_report = state_space.identify_cycles()

print(f"Total Valid Transitions: {cycle_report['total_transitions']}")
print(f"Self-Loop Cyclic Transitions: {cycle_report['total_self_loops']}")
print(f"State-Changing Transitions: {cycle_report['state_changing_transitions']}")
print("\nSelf-Loops broken down by action:")
for action, count in cycle_report["self_loops_by_action"].items():
    print(f"  - Action '{action}': {count} self-loops")

print("\nIdentified Self-Loop Transitions:")
for rec in cycle_report["self_loop_records"]:
    print(f"  {rec.source.short_label} + '{rec.action}' -> {rec.description}")

### 5. Optimal Search: Shortest Paths to Goal (BFS)
Using Breadth-First Search (BFS), we compute the minimal number of steps and the exact action sequence required to transform each state into a clean goal state.

In [ ]:
print(f"{'Start State':<18}{'Steps':<8}{'Action Sequence':<30}{'Goal State'}")
print("-" * 70)
for s in state_space.states:
    res = state_space.find_shortest_paths_to_goal(s)
    actions_str = " -> ".join(res["actions"]) if res["actions"] else "(Already Clean)"
    print(f"{s.short_label:<18}{res['distance']:<8}{actions_str:<30}{res['goal'].short_label}")

### 6. State-Space Directed Graph Visualization (NetworkX + Matplotlib)

- **Layout**: Structured 2-column textbook layout (Location A on the left, Location B on the right, ordered vertically by cleanliness level).
- **Optional Challenge**: Goal states (States 7 and 8) are highlighted in **Emerald Green** (`#2ecc71`).
- **Curved Directed Edges**: State-changing actions are rendered with curved directed arrows to prevent reciprocal edge overlap.
- **Self-Loop Badges**: Cyclic transitions are marked with dashed self-loop indicators.

In [ ]:
# Render publication-grade figure
fig, ax = visualizer.plot_graph(
    layout_type="textbook",
    highlight_goals=True,
    highlight_cycles=True,
    show=False,
)
plt.show()

### 7. Alternative Graph Layouts: Circular & Spring
Demonstrating alternative topological views of the state space graph.

In [ ]:
# Render circular layout
fig_circ, ax_circ = visualizer.plot_graph(
    layout_type="circular",
    highlight_goals=True,
    highlight_cycles=True,
    title="State-Space Graph: Circular Topology",
    show=False,
)
plt.show()

### 8. Adjacency Matrix Representation ($O(|S|^2)$ Space)

In [ ]:
import pandas as pd

labels, matrix = state_space.to_adjacency_matrix()
adj_df = pd.DataFrame(matrix, index=labels, columns=labels)
adj_df

### 9. Viva Voce Theory & Reference Guide

In [ ]:
print(get_viva_reference())